<a href='https://colab.research.google.com/github/Emelecto/QuantLab/blob/main/web/content/cursos/riesgo/notebooks/c4_l7.ipynb' target='_parent'><img src='https://colab.research.google.com/assets/colab-badge.svg'/></a>

# C4-L7 · Plan auditable | 50 trades con y sin plan: winrate 63% vs 30%, +0,63R vs −0,21R.

In [ ]:
# CELDA COLAB-FIRST: correla primero si estas en Google Colab.
# Descarga el CSV del repo; si falla (sin red ), usa el CSV local.
import pandas as pd
from pathlib import Path

ORG = "Emelecto"  # organizacion fija del repo Emelecto/QuantLab
CSV_NOMBRE = "c4_l7.csv"
CSV_URL = f"https://raw.githubusercontent.com/{ORG}/QuantLab/main/web/content/cursos/riesgo/data/{CSV_NOMBRE}"

try:
    df = pd.read_csv(CSV_URL)
    print("CSV descargado desde:", CSV_URL)
except Exception as e:
    print("Uso CSV local (motivo:", str(e)[:80] + ")")
    csv_path = Path("../data") / CSV_NOMBRE
    if not csv_path.exists():
        csv_path = Path(CSV_NOMBRE)  # fallback si corres desde data/
    df = pd.read_csv(csv_path)
print(df.shape)
print(df.head())

In [ ]:
import numpy as np
for grupo in ["si", "no"]:
    sub = df[df["tiene_plan"] == grupo]
    wr = (sub["resultado"] == "win").mean()
    print("Plan=%s: n=%d winrate=%.1f%% R_prom=%+.2fR" % (grupo, len(sub), 100 * wr, sub["r_multiple"].mean()))

## La R estandariza todo | 1R es tu riesgo por trade: +0,63R significa ganar en promedio el 63% de lo arriesgado.

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df[df["tiene_plan"] == "si"]["r_multiple"], bins=10, alpha=0.7, color="#5eead4", label="con plan")
ax.hist(df[df["tiene_plan"] == "no"]["r_multiple"], bins=10, alpha=0.7, color="#f87171", label="sin plan")
ax.axvline(0, color="#52525b", linewidth=1)
ax.set_xlabel("R por trade")
ax.legend()
plt.show()

In [ ]:
print("Brecha: el grupo con plan filtra entradas y estandariza el riesgo.")
print("Un ganador sin plan cuenta como FALLO de proceso.")

In [ ]:
# Chequeos automáticos
assert len(df) == 50, "se esperan 50 trades"
con = df[df["tiene_plan"] == "si"]; sin_ = df[df["tiene_plan"] == "no"]
assert len(con) == 30 and len(sin_) == 20, "30 con plan y 20 sin plan"
assert abs((con["resultado"] == "win").mean() - 19/30) < 1e-9, "winrate con plan 63%"
assert abs((sin_["resultado"] == "win").mean() - 0.30) < 1e-9, "winrate sin plan 30%"
assert abs(con["r_multiple"].mean() - 0.63) < 0.01, "R con plan +0,63"
assert abs(sin_["r_multiple"].mean() - (-0.21)) < 0.01, "R sin plan -0,21"
print("OK: el plan paga.")